# Municipality types of Nepal

Assigns each municipality in `nepal_municipalities` its municipality type:

- [List of Municipalities of Nepal](https://nepalog.com/state-structural-data/list-of-municipalities-of-nepal/) → Metropolitan City (6), Sub-Metropolitan City (11), Urban Municipality (276)
- [List of Rural Municipalities in Nepal](https://nepalog.com/state-structural-data/list-of-rural-municipalities-in-nepal/) → Rural Municipality (460)

Result: new column `municipality_type` in `nepal_municipalities`. If the assignment is not unambiguous, the value is left empty (NULL).

## 1. Setup and database connection

In [2]:
import os
import re
from difflib import SequenceMatcher
from html.parser import HTMLParser

import requests
import pandas as pd
import geopandas as gpd
from sqlalchemy import create_engine, text
from dotenv import load_dotenv

load_dotenv()  # reads variables from .env (never committed / shared)

db_user = os.getenv("DB_USER")
db_password = os.getenv("DB_PASSWORD")
db_host = os.getenv("DB_HOST", "localhost")
db_port = os.getenv("DB_PORT", "5432")
db_name = os.getenv("DB_NAME")

engine = create_engine(
    f"postgresql://{db_user}:{db_password}@{db_host}:{db_port}/{db_name}"
)

## 2. Load municipalities

Besides `name`, `name:en`, name suffixes (`name:suffix`) and alternative names are taken from `tags` (hstore). The district of each municipality is determined spatially: representative point of the municipality → district boundary (`planet_osm_polygon`, `admin_level = 6`). Name + district separate municipalities with the same name (e.g. Aurahi, Kalika, Rajpur).

In [3]:
municipalities = gpd.read_postgis(
    """
    SELECT *,
           tags -> 'name:en'        AS name_en,
           tags -> 'name:suffix'    AS name_suffix,
           tags -> 'name:suffix:en' AS name_suffix_en,
           tags -> 'alt_name:en'    AS alt_name_en,
           tags -> 'old_name:en'    AS old_name_en,
           tags -> 'short_name:en'  AS short_name_en,
           tags -> 'alt_name'       AS alt_name,
           tags -> 'old_name'       AS old_name,
           tags -> 'short_name'     AS short_name
    FROM nepal_municipalities
    """,
    engine,
    geom_col="way",
).drop(columns="municipality_type", errors="ignore")  # written by section 7 in a previous run

districts = gpd.read_postgis(
    """
    SELECT osm_id AS district_osm_id, COALESCE(tags -> 'name:en', name) AS district, way
    FROM planet_osm_polygon
    WHERE boundary = 'administrative' AND admin_level = '6'
    """,
    engine,
    geom_col="way",
)

rep_points = gpd.GeoDataFrame(geometry=municipalities.representative_point(), crs=municipalities.crs)
joined = gpd.sjoin(rep_points, districts, how="left", predicate="within")
municipalities["district"] = joined.loc[~joined.index.duplicated(), "district"]

print("Municipalities:", municipalities.shape, "| Districts:", len(districts))
print("Municipalities without district:", municipalities["district"].isna().sum())

Municipalities: (752, 80) | Districts: 77
Municipalities without district: 0


## 3. Municipality lists from nepalog.com

Tables are parsed with `html.parser` (standard library) – `lxml`/`bs4` are not installed in osm_env. On the municipality page, the type follows from the heading above each table; on the rural page, everything is Rural Municipality. The column positions (name, district) are read from the header row, as they change between tables.

Note on the source: the rural list contains "Rhishing" (Tanahun) twice (no. 439 and 441) – after removing the duplicate, 459 instead of 460 entries remain. As a result, Ghiring (Tanahun) is missing from the source and stays empty. Nepalgunj, Butwal and Duduwa are listed in the source but missing from `nepal_municipalities`.

In [4]:
SOURCES = {
    "urban": "https://nepalog.com/state-structural-data/list-of-municipalities-of-nepal/",
    "rural": "https://nepalog.com/state-structural-data/list-of-rural-municipalities-in-nepal/",
}


class TableParser(HTMLParser):
    """Collects all <table>s on the page, each with the last heading (h2/h3) before it."""

    def __init__(self):
        super().__init__()
        self.tables, self.heading = [], None
        self._in_heading = self._in_cell = False
        self._buf, self._row, self._rows = "", None, None

    def handle_starttag(self, tag, attrs):
        if tag in ("h2", "h3"):
            self._in_heading, self._buf = True, ""
        elif tag == "table":
            self._rows = []
        elif tag == "tr" and self._rows is not None:
            self._row = []
        elif tag in ("td", "th") and self._row is not None:
            self._in_cell, self._buf = True, ""

    def handle_endtag(self, tag):
        if tag in ("h2", "h3") and self._in_heading:
            self.heading, self._in_heading = " ".join(self._buf.split()), False
        elif tag in ("td", "th") and self._in_cell:
            self._row.append(" ".join(self._buf.split()))
            self._in_cell = False
        elif tag == "tr" and self._row is not None:
            self._rows.append(self._row)
            self._row = None
        elif tag == "table" and self._rows is not None:
            self.tables.append((self.heading, self._rows))
            self._rows = None

    def handle_data(self, data):
        if self._in_heading or self._in_cell:
            self._buf += data


def type_from_heading(page, heading):
    if page == "rural":
        return "Rural Municipality"
    h = (heading or "").lower()
    if "sub-metropolitan" in h:
        return "Sub-Metropolitan City"
    if "metropolitan" in h:
        return "Metropolitan City"
    if "urban municipality" in h:
        return "Urban Municipality"
    return None


records = []
for page, url in SOURCES.items():
    response = requests.get(url, headers={"User-Agent": "Mozilla/5.0"}, timeout=60)
    response.raise_for_status()
    parser = TableParser()
    parser.feed(response.text)

    col_idx = None  # (name, district) – carries over to following tables without a header row
    for heading, rows in parser.tables:
        municipality_type = type_from_heading(page, heading)
        if municipality_type is None:
            continue
        for row in rows:
            if not row[0].rstrip(".").isdigit():  # header row
                low = [c.lower() for c in row]
                d = next(i for i, c in enumerate(low) if "district" in c)
                n = next(i for i, c in enumerate(low) if i not in (0, d)
                         and any(k in c for k in ("name", "municipal", "metropolitan", "city")))
                col_idx = (n, d)
                continue
            n, d = col_idx
            records.append({"src_name": row[n], "src_district": row[d],
                            "municipality_type": municipality_type})

source = (pd.DataFrame(records)
          .drop_duplicates(subset=["src_name", "src_district", "municipality_type"])
          .reset_index(drop=True))
source["municipality_type"].value_counts()

municipality_type
Rural Municipality       459
Urban Municipality       276
Sub-Metropolitan City     11
Metropolitan City          6
Name: count, dtype: int64

## 4. Normalise names

Transliterations differ between OSM and the source (Tribeni/Triveni, Menchayayem/Menchhayayem, Arjunchaupari/Arjun Chaupari). Names are therefore reduced to a "skeleton": lower case, type suffixes (Gaunpalika, Municipality, …) removed, Latin letters only, aspirated consonants (bh, chh, dh, …) simplified, v/w → b, sh → s, aa/ee/oo → a/i/u, double letters collapsed. For districts, the word order is also sorted (Western Rukum = Rukum West).

In addition, a **type hint** is derived from `name:suffix` or the name (Gaunpalika → Rural, Nagarpalika → Urban, …). It is only used to resolve ambiguous matches and as a plausibility check – the assigned type always comes from the source.

In [5]:
TYPE_WORDS = (r"\b(sub[- ]?metropolitan|metropolitan|city|rural|municipality|municipalities|"
              r"gaunpalika|gaupalika|nagarpalika|mahanagarpalika|upa[- ]?mahanagarpalika|district)\b")


def is_latin(s):
    return isinstance(s, str) and re.search(r"[A-Za-z]", s) is not None


def skeleton(s, sort_words=False):
    s = re.sub(r"\(.*?\)", " ", s.lower())  # bracketed additions such as "(Panchthar)", "(Nepal)"
    s = s.replace("western", "west").replace("eastern", "east")
    s = re.sub(TYPE_WORDS, " ", s)
    words = re.findall(r"[a-z]+", s)
    if sort_words:
        words = sorted(words)
    s = "".join(words)
    for a, b in [("aa", "a"), ("ee", "i"), ("oo", "u"), ("w", "b"), ("v", "b"), ("sh", "s"),
                 ("ts", "c"), ("z", "j"), ("f", "p"), ("y", "i")]:
        s = s.replace(a, b)
    for _ in range(2):  # chh -> ch -> c
        s = re.sub(r"([bcdgjkpt])h", r"\1", s)
    return re.sub(r"(.)\1+", r"\1", s)


def similarity(a, b):
    return SequenceMatcher(None, a, b).ratio()


def name_similarity(a, b):
    """Like similarity, but a fully contained name (Sulikot in Barpak Sulikot) counts at least 0.9."""
    short, long_ = sorted((a, b), key=len)
    if len(short) >= 4 and short != long_ and short in long_:
        return max(0.9, similarity(a, b))
    return similarity(a, b)


# Order matters: "उपमहानगर" contains "महानगर", "Rural Municipality" contains "municipality"
TYPE_KEYWORDS = [
    ("Sub-Metropolitan City", ("upamahanagar", "upa-mahanagar", "उपमहानगर", "उप-महानगर", "sub-metropolitan", "sub metropolitan")),
    ("Metropolitan City", ("mahanagar", "महानगर", "metropolitan")),
    ("Rural Municipality", ("gaunpalika", "गाउँपालिका", "rural")),
    ("Urban Municipality", ("nagarpalika", "नगरपालिका", "municipality")),
]


def type_of_text(s):
    s = s.lower()
    return next((t for t, keys in TYPE_KEYWORDS if any(k in s for k in keys)), None)


def type_hint(row):
    """Type according to the OSM name suffix or name – if the fields contradict each other, there is no hint."""
    hints = {type_of_text(v) for v in row[["name_suffix", "name_suffix_en", "name", "name_en"]] if isinstance(v, str)}
    hints.discard(None)
    return hints.pop() if len(hints) == 1 else None


NAME_COLS = ["name_en", "name", "alt_name_en", "old_name_en", "short_name_en", "alt_name", "old_name", "short_name"]
municipalities["name_keys"] = municipalities[NAME_COLS].apply(
    lambda r: sorted({skeleton(v) for v in r if is_latin(v)} - {""}), axis=1
)
municipalities["type_hint"] = municipalities.apply(type_hint, axis=1)
source["name_key"] = source["src_name"].apply(skeleton)
# The source calls Nawalparasi West "Parasi" in some of its entries
DISTRICT_ALIASES = {"Parasi": "Nawalparasi West"}
source["district_key"] = source["src_district"].replace(DISTRICT_ALIASES).apply(lambda d: skeleton(d, sort_words=True))

municipalities[["name", "name_en", "name_keys", "district", "type_hint"]].head()

,name,name_en,name_keys,district,type_hint
0,Dunhu,NaN,[dunhu],Darchula District (Nepal),Rural Municipality
1,Mahakali,NaN,[mahakali],Kanchanpur District,Urban Municipality
2,आठराई गाउँपालिका,Aathrai,[atrai],Tehrathum,Rural Municipality
3,फेदाप गाउँपालिका,Phedap,[pedap],Tehrathum,Rural Municipality
4,चैनपुर नगरपालिका,Chainpur,[cainpur],Sankhuwasabha,Urban Municipality


### Match districts

OSM district names are mapped to the most similar district in the source (similarity ≥ 0.8). The table shows all non-exact mappings for checking.

In [6]:
DISTRICT_MIN = 0.8
src_district_keys = source["district_key"].unique()


def best_district(district):
    if not is_latin(district):
        return None, 0.0
    key = skeleton(district, sort_words=True)
    scores = {k: similarity(key, k) for k in src_district_keys}
    best = max(scores, key=scores.get)
    return best, scores[best]


district_map = pd.DataFrame({"district": municipalities["district"].dropna().unique()})
district_map[["src_district_key", "district_score"]] = district_map["district"].apply(
    lambda d: pd.Series(best_district(d))
)
district_map.loc[district_map["district_score"] < DISTRICT_MIN, "src_district_key"] = None

municipalities = municipalities.merge(district_map[["district", "src_district_key"]], on="district", how="left")
district_map[district_map["district_score"] < 1].sort_values("district_score")

,district,src_district_key,district_score
47,Nawalparasi W,nabalparasibest,0.888889
2,Tehrathum,teratum,0.933333
11,Sindhupalchowk,sindupalcok,0.956522


## 5. Matching

For each municipality:

1. Candidates from the source with name similarity ≥ 0.85 (best value across all name variants; a fully contained name such as *Sulikot* in *Barpak Sulikot* counts as 0.9).
2. If there are candidates **in the same district**, only those with the highest similarity count. Otherwise: only **identical** name skeletons nationwide.
3. If the remaining candidates have different types, the OSM type hint decides – if that is not unambiguous → empty.
4. Also left empty: a fuzzy match (< 1.0) or a nationwide match whose type contradicts the OSM hint.
5. If the same source entry is matched by several different OSM municipalities, only the best match keeps it; the others stay empty. In case of a tie, the match whose OSM type hint agrees with the source wins (e.g. Gorkha municipality vs. a fragment with the same name but without `boundary`).

In [7]:
NAME_MIN = 0.85


def match(row):
    if not row["name_keys"]:
        return None, None, 0.0, "no Latin name"
    scores = source["name_key"].apply(lambda k: max(name_similarity(n, k) for n in row["name_keys"]))
    cands = source.assign(score=scores)[scores >= NAME_MIN]
    hint = row["type_hint"]

    same_district = cands[cands["district_key"] == row["src_district_key"]]
    if len(same_district):
        top, scope = same_district[same_district["score"] == same_district["score"].max()], "district"
    else:
        top, scope = cands[cands["score"] == 1.0], "nationwide"
    if top.empty:
        return None, None, 0.0, "no match"

    if top["municipality_type"].nunique() > 1:
        top = top[top["municipality_type"] == hint]
        if top["municipality_type"].nunique() != 1:
            return None, None, 0.0, f"ambiguous ({scope})"

    municipality_type, score = top["municipality_type"].iloc[0], top["score"].iloc[0]
    if hint is not None and hint != municipality_type and (score < 1.0 or scope == "nationwide"):
        return None, None, score, f"type conflict ({scope}, fuzzy)" if score < 1.0 else f"type conflict ({scope})"
    src_idx = top.index[0] if len(top) == 1 else None
    return municipality_type, src_idx, score, scope


result = municipalities.apply(match, axis=1, result_type="expand")
result.columns = ["municipality_type", "src_idx", "match_score", "match_info"]
municipalities = municipalities.join(result)

# Same source entry for several different OSM municipalities -> keep only the clearly best match
for src_idx, grp in municipalities.dropna(subset=["src_idx"]).groupby("src_idx"):
    if grp["osm_id"].nunique() > 1:
        winners = grp[grp["match_score"] == grp["match_score"].max()]
        if winners["osm_id"].nunique() > 1:  # tie: agreement with the OSM type hint decides
            winners = winners[winners["type_hint"] == winners["municipality_type"]]
        losers = grp.index.difference(winners.index) if winners["osm_id"].nunique() == 1 else grp.index
        municipalities.loc[losers, "municipality_type"] = None
        municipalities.loc[losers, "match_info"] = "source entry assigned twice"

# Parts of the same OSM relation (same osm_id) must have the same type, otherwise empty
inconsistent = municipalities.groupby("osm_id")["municipality_type"].transform(lambda s: s.nunique(dropna=False) > 1)
municipalities.loc[inconsistent, "municipality_type"] = None
municipalities.loc[inconsistent, "match_info"] = "inconsistent parts"

municipalities["municipality_type"].value_counts(dropna=False)

municipality_type
Rural Municipality       442
Urban Municipality       274
NaN                       21
Sub-Metropolitan City      9
Metropolitan City          6
Name: count, dtype: int64

### Elimination per district

Some municipalities still carry old names in OSM (e.g. Sukidaha, Kanda) or are transliterated very differently (Bogtan / Bogatan Phudsil). If a district has as many unmatched OSM municipalities as unmatched source entries, all of these source entries have **the same type**, and the OSM type hint of each of these municipalities agrees with it, the type is unambiguous – even without knowing the exact pairing.

In [8]:
matched_src = set(municipalities.loc[municipalities["municipality_type"].notna(), "src_idx"].dropna().astype(int))
open_src = source[~source.index.isin(matched_src)]
open_rows = municipalities[municipalities["match_info"] == "no match"]

for district_key, grp in open_rows.groupby("src_district_key"):
    src_left = open_src[open_src["district_key"] == district_key]
    if grp["osm_id"].nunique() != len(src_left) or src_left["municipality_type"].nunique() != 1:
        continue
    municipality_type = src_left["municipality_type"].iloc[0]
    if (grp["type_hint"] == municipality_type).all():
        municipalities.loc[grp.index, "municipality_type"] = municipality_type
        municipalities.loc[grp.index, "match_info"] = "elimination"

municipalities.loc[municipalities["match_info"] == "elimination",
                   ["osm_id", "name", "district", "type_hint", "municipality_type"]]

,osm_id,name,district,type_hint,municipality_type
151,-10515187,Painyu,Parbat,Rural Municipality,Rural Municipality
252,-10470790,Kanda,Bajhang,Rural Municipality,Rural Municipality
265,-10507520,Suwarnabati,Rolpa,Rural Municipality,Rural Municipality
282,-12421492,Pandav Gupha,Bajura,Rural Municipality,Rural Municipality
399,-10534406,Bungdikali,Nawalpur District,Rural Municipality,Rural Municipality
491,-10516303,Narphu,Manang,Rural Municipality,Rural Municipality
493,-10516302,Neshyang,Manang,Rural Municipality,Rural Municipality
518,-10507528,Sukidaha,Rolpa,Rural Municipality,Rural Municipality
520,-10507525,Duikholi,Rolpa,Rural Municipality,Rural Municipality
532,-10507082,Kukmakhmalika,Salyan,Rural Municipality,Rural Municipality


## 6. Checks

In [9]:
print(municipalities["municipality_type"].value_counts(dropna=False).to_string(), "\n")
print(municipalities["match_info"].value_counts(dropna=False).to_string())

diag_cols = ["osm_id", "name", "name_en", "district", "type_hint", "municipality_type", "match_score", "match_info"]
print("\nMunicipalities left empty:")
municipalities.loc[municipalities["municipality_type"].isna(), diag_cols]

municipality_type
Rural Municipality       457
Urban Municipality       275
Sub-Metropolitan City      9
Metropolitan City          6
NaN                        5 

match_info
district                           725
elimination                         16
nationwide                           6
no match                             3
type conflict (district, fuzzy)      1
source entry assigned twice          1

Municipalities left empty:


,osm_id,name,name_en,district,type_hint,municipality_type,match_score,match_info
0,-10468413,Dunhu,NaN,Darchula District (Nepal),Rural Municipality,NaN,0.000000,no match
7,-10491218,Tribeni Nalagad,NaN,Jajarkot,NaN,NaN,0.000000,no match
101,-10468412,Byash,Byash Municipality,Darchula District (Nepal),NaN,NaN,0.888889,"type conflict (district, fuzzy)"
153,-12405040,Ghiring,NaN,Tanahun,Rural Municipality,NaN,0.000000,no match
605,-10468415,Mahakali,NaN,Darchula District (Nepal),Rural Municipality,NaN,1.000000,source entry assigned twice


In [10]:
# Assigned, but the type from the source differs from the OSM name suffix (exact name in the same district)
conflicts = municipalities[
    municipalities["municipality_type"].notna()
    & municipalities["type_hint"].notna()
    & (municipalities["municipality_type"] != municipalities["type_hint"])
]
conflicts[diag_cols]

,osm_id,name,name_en,district,type_hint,municipality_type,match_score,match_info
102,-10470787,Bungal,NaN,Bajhang,Rural Municipality,Urban Municipality,1.0,district


In [11]:
# Source entries that were not assigned to any OSM municipality
matched_src = set(municipalities["src_idx"].dropna().astype(int))
source.loc[~source.index.isin(matched_src), ["src_name", "src_district", "municipality_type"]]

,src_name,src_district,municipality_type
12,Nepalgunj,Banke,Sub-Metropolitan City
13,Butwal,Rupandehi,Sub-Metropolitan City
102,Nalgad,Jajarkot,Urban Municipality
127,Dodhara Chandani,Kanchanpur,Urban Municipality
179,Bardaghat,Nawalparasi West,Urban Municipality
323,Saipal,Bajhang,Rural Municipality
327,Jagannath,Bajura,Rural Municipality
333,Duduwa,Banke,Rural Municipality
377,Duhun,Darchula,Rural Municipality
378,Byans,Darchula,Rural Municipality


## 7. Update `nepal_municipalities`

The column `municipality_type` is added to the table in place (or overwritten if it already exists). This keeps the geometry (SRID 3857), `tags` as hstore and all other columns unchanged – a `to_postgis(if_exists="replace")` could not write the hstore column back.

In [12]:
type_by_osm_id = (
    municipalities.groupby("osm_id")["municipality_type"].first().reset_index()
)

with engine.begin() as conn:
    type_by_osm_id.to_sql("tmp_municipality_type", conn, if_exists="replace", index=False)
    conn.execute(text("ALTER TABLE nepal_municipalities ADD COLUMN IF NOT EXISTS municipality_type text"))
    conn.execute(text("UPDATE nepal_municipalities SET municipality_type = NULL"))
    conn.execute(text(
        "UPDATE nepal_municipalities t SET municipality_type = m.municipality_type "
        "FROM tmp_municipality_type m WHERE t.osm_id = m.osm_id"
    ))
    conn.execute(text("DROP TABLE tmp_municipality_type"))

pd.read_sql(
    "SELECT municipality_type, COUNT(*) AS n FROM nepal_municipalities GROUP BY 1 ORDER BY 2 DESC",
    engine,
)

,municipality_type,n
0,Rural Municipality,457
1,Urban Municipality,275
2,Sub-Metropolitan City,9
3,Metropolitan City,6
4,NaN,5
